# WeatherGPT — Kaggle T4 Training Notebook

Runs 3 pipelines: **semantic classifier → bias correction → intent parser**.
Works on **T4 x2** (fast) or **CPU** (slower). No data upload needed — synthetic fallback.

**Settings:** `Accelerator: GPU T4 x2`, Internet: On

```
!git clone https://github.com/<you>/weathergpt.git
%cd weathergpt
```
or upload this repo as a Kaggle Dataset and mount it.

In [ ]:
# 0 — Env check
import torch, platform, sys
print(f"Python {platform.python_version()}  torch {torch.__version__}")
print(f"cuda available: {torch.cuda.is_available()}  count: {torch.cuda.device_count() if torch.cuda.is_available() else 0}")
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
    try:
        print(f"T4 x2? count={torch.cuda.device_count()}")
    except: pass
!nvidia-smi 2>&1 | head -n 20

In [ ]:
# 1 — Install (skip if already installed)
# On Kaggle, torch+cuda is preinstalled. This just ensures FastAPI/transformers.
!pip install -q -r requirements.txt
# For full decoders (GRIB/HDF5) uncomment:
# !pip install -q -r requirements-kaggle.txt
print("install done")

In [ ]:
# 2 — Dry-run smoke (2s each, no HF download) — uncomment to test before full train
# !python training/train_semantic_classifier.py --dry-run
# !python training/train_bias_correction.py --dry-run
# !python training/train_intent_parser.py --dry-run
print("dry-run optional — skip for full train")

In [ ]:
# 3 — Train Semantic Variable Classifier (distilbert, ~2 min on T4, ~5 min CPU)
# Increase --epochs 5→10 for better score. Uses synthetic + training/datasets/field_names.csv if you uploaded one.
!python training/train_semantic_classifier.py --epochs 5 --batch-size 32 --device auto
print("semantic done — check training/models/semantic_classifier/metrics.json")

In [ ]:
# 4 — Train Bias-Correction MLP (GFS→AWS, ~3 min T4)
# Try --model lgbm for LightGBM variant (cpu, no torch needed).
!python training/train_bias_correction.py --model mlp --epochs 20 --batch-size 256 --device auto
print("bias correction done — check training/models/bias_correction/metrics.json")

In [ ]:
# 5 — Train Intent Parser (decision classifier, ~2 min T4)
!python training/train_intent_parser.py --epochs 3 --batch-size 32 --device auto
print("intent done — check training/models/intent_parser/metrics.json")

In [ ]:
# 6 — Collect artifacts for download
import pathlib, json, os
out = pathlib.Path("/kaggle/working/weathergpt_outputs")
out.mkdir(exist_ok=True)
!cp -r training/models "$out/" 2>&1 | head
!ls -lh training/models/*/* 2>&1 | head -n 50
for p in pathlib.Path("training/models").rglob("metrics.json"):
    print(f"\n== {p} ==")
    print(p.read_text())

In [ ]:
# 7 — Quick inference demo: load bias model + correct a sample
import torch, json, pathlib
from training.train_bias_correction import MLP
import numpy as np
p = pathlib.Path("training/models/bias_correction/best.pt")
if p.exists():
    cfg = json.loads(pathlib.Path("training/models/bias_correction/config.json").read_text())
    m = MLP(in_dim=cfg["in_dim"], hidden=cfg["hidden"], layers=cfg["layers"], out_dim=cfg["out_dim"])
    sd = torch.load(str(p), map_location="cpu")
    # handle DataParallel prefix
    sd = {k.replace("module.",""):v for k,v in sd.items()}
    m.load_state_dict(sd)
    m.eval()
    # sample: gfs_t2m_norm=0.5, apcp=0.2, elev=0.1, lead=0.3, lat=0.2
    x = torch.tensor([[0.5,0.2,0.1,0.3,0.2]])
    bias = m(x)
    print(f"sample bias (t2m, apcp): {bias.detach().numpy()}")
else:
    print("no model yet — run cell 4")

In [ ]:
# 8 — Backend smoke (no keys, mock WIO) — optional in Kaggle
# !uvicorn app.main:app --port 8001 &
# !sleep 3 && curl -s -X POST http://localhost:8001/wio/query -H "Content-Type: application/json" -d '{"question":"Will it rain in Nagpur tomorrow afternoon?","location":{"raw":"Nagpur"},"lang":"en"}' | head -c 2000